# 07 — Threads, Locks, Queues, and Shared State

Threads are useful for blocking I/O, but concurrent access to mutable state needs synchronization. Learn locks, queues, bounded pools, and the limits of the GIL.

*Advanced · 90–150 min*

## You will learn
- Submit tasks using ThreadPoolExecutor
- Use Lock for shared mutable state
- Communicate via queue.Queue
- Avoid assuming the GIL makes all compound operations safe

In [ ]:
# Setup: run this first. On Google Colab (or anywhere the course files are missing) it
# downloads the course and moves into this notebook's folder. Elsewhere it does nothing.
import os, sys
if not os.path.exists("../README.md"):
    course = os.path.join(os.path.expanduser("~"), "python-notebook")
    if not os.path.isdir(course):
        os.system(f"git clone -q --depth 1 https://github.com/SwarmLadder/python-notebook {course}")
    os.chdir(os.path.join(course, "advanced", "notebooks"))
    sys.path.insert(0, os.getcwd())

## Examples

### 1. Thread pools and results

A ThreadPoolExecutor manages a bounded set of worker threads. Use a with context so the pool shuts down and completes submitted work. Map preserves input order; futures offer completion-based APIs.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def square(n):
    return n * n

with ThreadPoolExecutor(max_workers=3) as pool:
    values = list(pool.map(square, [1, 2, 3, 4]))
print(values)

**Think:** Why can the ordering of individual completion times differ from map output order?

### 2. Protecting shared state

A lock makes critical sections mutually exclusive. Even if individual operations are atomic on a specific interpreter, read-modify-write sequences should not rely on that. Prefer local state and message passing when practical.

In [ ]:
import threading
from concurrent.futures import ThreadPoolExecutor

counter = {"value": 0}
lock = threading.Lock()

def increment_many(times):
    for _ in range(times):
        with lock:
            counter["value"] += 1

with ThreadPoolExecutor(max_workers=4) as pool:
    list(pool.map(increment_many, [100] * 4))
print(counter)
assert counter["value"] == 400

**Think:** What race might occur between reading, adding one, and writing without synchronization?

### 3. Queues communicate work

queue.Queue is a thread-safe producer–consumer channel. A bounded maxsize creates backpressure. Each submitted item should receive a corresponding task_done call if consumers use queue.join.

In [ ]:
from queue import Queue
import threading

q = Queue(maxsize=3)
results = []

def consume():
    while True:
        item = q.get()
        try:
            if item is None:
                return
            results.append(item * 2)
        finally:
            q.task_done()

thread = threading.Thread(target=consume)
thread.start()
for value in (1, 2, 3):
    q.put(value)
q.put(None)
q.join()
thread.join()
print(results)

**Think:** Why is a bounded queue valuable during overload?

### 4. Thread safety of APIs

Threading introduces nondeterministic scheduling, so tests should assert invariants rather than an arbitrary execution order. Protect shared data, avoid holding locks while doing slow I/O, and establish consistent lock ordering to prevent deadlocks.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def label(n):
    return f"task-{n}"

with ThreadPoolExecutor(max_workers=2) as pool:
    futures = [pool.submit(label, n) for n in range(4)]
    labels = {future.result() for future in as_completed(futures)}
print(sorted(labels))
assert len(labels) == 4

**Think:** What changes in tests when result ordering is not guaranteed?

## Try it

Short fill-in exercises on this lesson: replace each `___`, run the cell, and `check` tells you whether you're right. The answer itself stays hidden. More practice for this topic is in the [workbook](../exercises/02_concurrency_resilience_exercises.ipynb).

In [ ]:
from lesson import Exercises, ___, run

check = Exercises('advanced/07', {1: 'd7f7fecd69c30e6b', 2: '4be892a404b29f8e', 3: '1a7c851eb9112160', 4: '9cc05fb9b5432c49'})

**Exercise 1.** Four threads each add 10,000 to a shared counter, guarded by a lock. Predict the exact total.

In [ ]:
import threading

total = 0
lock = threading.Lock()

def add_many():
    global total
    for _ in range(10000):
        with lock:              # without the lock, updates could be lost
            total += 1

threads = [threading.Thread(target=add_many) for _ in range(4)]
for t in threads: t.start()
for t in threads: t.join()
expected = ___
check(1, (total, total == expected))

**Exercise 2.** A producer/consumer pipeline with a bounded queue and a `None` sentinel.

In [ ]:
import queue, threading

q = queue.Queue(maxsize=2)
results = []

def consumer():
    while (item := q.get()) is not None:
        results.append(item.upper() if isinstance(item, str) else item)

t = threading.Thread(target=consumer, daemon=True)
t.start()
for word in ["a", "b", "c"]:
    q.put(word)
q.put(___)                     # the sentinel that tells the consumer to stop
t.join(timeout=1)
check(2, (results, t.is_alive()))

**Exercise 3.** Collect results as they complete, but keep track of which input each came from.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

def slow_square(n):
    time.sleep(0.01 * (4 - n))
    return n * n

with ThreadPoolExecutor(max_workers=4) as pool:
    futures = {pool.submit(slow_square, n): n for n in range(4)}
    results = {futures[f]: f.___ for f in as_completed(futures)}
check(3, results)

**Exercise 4.** Pick the right tool for each workload: `asyncio`, `threads` or `processes`.

In [ ]:
choices = {
    "download 500 web pages": ___,
    "resize 10,000 images in pure Python": ___,
    "call a blocking library from async code": ___,
}
check(4, choices)

## Common mistakes

- The GIL does not guarantee correctness of compound shared-state operations.
- Use context managers for executors and locks.
- Do not rely on task completion order in concurrent tests.

## Quick quiz

Replace each `'?'` in the cell below with your choice and run it. The answers are hidden, but each one is checked.

**1.** Is `counter += 1` safe from several threads without a lock?

   - **a)** No, it's a read-modify-write that can interleave
   - **b)** Only on Linux
   - **c)** Yes, the GIL protects it

**2.** queue.Queue is useful because it…

   - **a)** Is thread-safe for producers and consumers
   - **b)** Runs code faster
   - **c)** Avoids threads

**3.** Threads help most with…

   - **a)** I/O-bound waiting
   - **b)** Neither
   - **c)** CPU-bound pure Python

In [ ]:
from quiz import Quiz

quiz = Quiz('advanced/07', {1: 'e7117ec4e8e6', 2: '35d5e60d3456', 3: '5a3855e387e1'})
quiz.answer(1, '?')
quiz.answer(2, '?')
quiz.answer(3, '?')

**Next:** [08 — Multiprocessing, Process Pools, and the GIL](08_multiprocessing_and_parallelism.ipynb)